**LANGCHAIN RAG CHATBOT:**

**INSTALLATION:**

In [ ]:
!pip install -q langchain langchain-classic langchain-groq langchain-community langchain-qdrant langchain-huggingface langchain-text-splitters pypdf
!pip install -q langchain-qdrant qdrant-client

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


**IMPORTING ALL NECESSARY LIBRARIES:**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import Qdrant
from langchain_qdrant import QdrantVectorStore
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_471/1793950109.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**STEP # 01: LOAD & SPLIT THE DOCUMENT**

In [ ]:
# Step : 1 PDF file
loader = PyPDFLoader("UGE-Policy.pdf")

# Step 2: Load PDF
docs = loader.load()

# Step 3: Print number of pages
print(f"Total pages: {len(docs)}")



Total pages: 18


In [ ]:
# Step : 1 Text splitting (chunks formation):
splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)
chunks = splitter.split_documents(docs)

# Step : 2 Chunks length
print(f"loaded {len(chunks)} chunks")

loaded 201 chunks


**STEP # 02 EMBEDDING & STORE IN VECTOR DATABASE**

In [ ]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store= QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="my_docs"
)
retriever= vector_store.as_retriever(search_kwargs={"k":5})


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**STEP # 03 BUILD THE PROMPT AND MODEL**

In [ ]:
from google.colab import userdata
api_key=userdata.get('ChatBotKey')
Client =Groq(api_key=api_key)


In [ ]:
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5,
    max_tokens=300,
    api_key=api_key
)
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Answer using ONLY the provided context.
If the answer is not in the context, reply exactly:
I don't know.

Context:
{context}"""
    ),

     MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

**STEP # 04 RAG ASSEMBLING**

In [ ]:
# Step 1: Create parser
parser = StrOutputParser()
doc_chain = create_stuff_documents_chain(
    model,
    prompt,
    output_parser=parser
)
chain = prompt | model | parser

# Create retrieval chain
rag_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=doc_chain
)

# Ask a question
response = rag_chain.invoke({
    "input": "Tell me Structure of Bachelor Degree Program?",
     "chat_history": []
})

# Print answer
print(response["answer"])


The structure of Bachelor degree programs includes:

- Nomenclature of the Degree Programs  
- Credit Hours  
- General Education Courses  
- Major Courses  
- Interdisciplinary/Allied Courses


**STEP # 05: HISTORY**

In [ ]:
store={}

# Step : 01 get_history
def get_history(session_id):
  if session_id not in store:
    store[session_id]=InMemoryChatMessageHistory()
  return store[session_id]

# Step : 02 conversational_rag
conversational_rag=RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"

)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


**STEP # 06 RUN FULL APP**

In [ ]:
config = {"configurable": {"session_id": "user_123"}}

print("*********** Welcome to HEC UGEP RAG Chatbot ***********")
print()

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("Chatbot: Goodbye!")
        break
    response = conversational_rag.invoke(
        {"input": user_input},
        config=config
    )
    print("Chatbot:", response["answer"])

*********** Welcome to HEC UGEP RAG Chatbot ***********

You: give me info about arts 


/tmp/ipykernel_471/1713000074.py:6: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id]=InMemoryChatMessageHistory()


Chatbot: Arts are included in the GENERAL EDUCATION REQUIREMENT under the Arts and Humanities category. The purpose of introducing a course in the domain of Arts (and Applied Knowledge) is to:

* Emphasize creativity – equipping students with the sense and ability to demonstrate creativity, curiosity, exploration, and reflective problem‑solving.
* Familiarize students with key concepts in disciplines such as Philosophy, History, Creative Arts, etc., helping them understand human experience.
* Provide exposure not only to the main field of specialization but also to interdisciplinary areas of knowledge.
You: tell me about associate degree program
Chatbot: An Associate Degree is a two‑year post‑higher‑secondary‑school certificate (HSSC) academic degree. It is offered by universities mainly in market‑driven subjects and is designed to provide fundamental academic and working knowledge in a particular field, preparing graduates for employment or further study.

**Key points about the progr